---

# Forecasting Pipeline Template

*An EDA-driven, global gradient boosting forecasting pipeline for one or many time series. Every modelling decision is read from `eda-output/eda_decisions.json`, the model can be switched between LightGBM, XGBoost, and CatBoost, statistical baselines are exported to `pipeline-output/baseline/`, and the final forecast to `pipeline-output/final/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Toolkit**](#3)
4. [**EDA-Driven Decisions**](#4)
5. [**Data Preparation**](#5)
6. [**Feature Engineering**](#6)
7. [**Backtesting Design**](#7)
8. [**Statistical Baselines**](#8)
9. [**Gradient Boosting Forecasters**](#9)
10. [**Ensemble and Model Selection**](#10)
11. [**Backtest Diagnostics**](#11)
12. [**Model Interpretation**](#12)
13. [**Final Forecast**](#13)
14. [**Export**](#14)


---

# Introduction <a name="1"></a>

---

## Overview

*Global gradient boosting models (one model trained on all series with lag, rolling, calendar, and event features) have won most recent forecasting competitions, including M5. They only work well when the feature design matches the data: the right lags for the horizon, the right transform, the right treatment of zeros and events. This notebook takes those choices from the forecasting EDA instead of guessing them.*

## Aim

*The notebook backtests naive, seasonal naive, moving average, and ETS baselines and any of LightGBM, XGBoost, and CatBoost on identical expanding-window folds, blends them, selects the best by backtest error, refits on the full history, and produces the future forecast with empirical prediction intervals and a submission file.*

## How to Configure

*Settings on `'auto'` are filled from `eda-output/eda_decisions.json` (written by `eda_forecasting.ipynb`) or computed from the data when the file is missing.*

| Setting | Purpose | Example |
|---|---|---|
| `*_PATH`, `TEST_*_PATH` | History file and future frame (dates plus known covariates) | `'train.csv'`, `'test.csv'` |
| `DATE_COL`, `TARGET_COL`, `ID_COLS` | Long-format columns | `'date'`, `'sales'`, `['store', 'item']` |
| `MODELS` | Any subset of `['lightgbm', 'xgboost', 'catboost']` | `['catboost']` |
| `STRATEGY` | `'direct'` (lags >= horizon, one pass) or `'recursive'` (short lags, step by step) | `'direct'` |
| `HORIZON`, `FREQ`, `SEASONAL_PERIODS`, `TRANSFORM`, `LAGS`, `ROLL_WINDOWS` | Forecast design, `'auto'` from EDA | `28`, `'D'`, `[7, 365]` |
| `OBJECTIVE`, `WEIGHT_BY_VOLUME` | Loss and sample weighting, `'auto'` from EDA | `'tweedie'`, `True` |
| `EVENT_DATES` | Known event dates (past and future), for example Eid al-Fitr | `['2025-03-31', ...]` |
| `N_FOLDS`, `METRIC` | Backtest folds and selection metric | `5`, `'wape'` |

*A model whose library is not installed is skipped, and the statistical baselines always run.*

## Metric

***WAPE** (weighted absolute percentage error) is the default when the series contain zeros, because it weights errors by volume and never divides by a zero actual. **MASE** scales the error by the in-sample seasonal naive error, so values below 1 beat seasonal naive. RMSE, sMAPE, and bias are reported alongside.*

$$\text{WAPE} = \frac{\sum_{i,t} |y_{i,t} - \hat{y}_{i,t}|}{\sum_{i,t} |y_{i,t}|} \qquad \text{MASE}_i = \frac{\frac{1}{h}\sum_{t} |y_{i,t} - \hat{y}_{i,t}|}{\frac{1}{n-m}\sum_{t=m+1}^{n} |y_{i,t} - y_{i,t-m}|}$$

## Dataset

*The demo uses the same synthetic retail panel as the forecasting EDA: daily sales for 6 stores x 5 items with weekly and yearly seasonality, Eid al-Fitr (Lebaran) peaks, promotions, prices, intermittent items, gaps, outliers, a late-opening store, and a level shift. The 28-day test frame carries future prices and promotions.*

```
demo-data/
├── train.csv   (date, store, item, sales, price, promo)
└── test.csv    (date, store, item, price, promo)
eda-output/
└── eda_decisions.json
```

## Approach: Global Gradient Boosting on EDA-Designed Features

*All series are stacked into one long table and a single model learns shared dynamics, which lets short and new series borrow strength from long ones. The direct strategy predicts every step of the horizon from information available at the forecast origin, so no prediction is fed back as an input and backtest error is an honest estimate of live error.*

```
eda_decisions.json --> FREQ, HORIZON, periods, transform, fill, lags, windows, exog, events, metric
Long history + future frame -> regular grid -> fill -> transform
    |
    +-- Features   : lags (>= horizon for direct), rolling mean/std, same-season mean, EWM,
    |                calendar, event distance, known covariates, series ids, series age
    +-- Backtest   : expanding-window folds of HORIZON steps
    |     +-- Baselines : naive, seasonal naive, moving average, ETS     -> pipeline-output/baseline/
    |     +-- GBMs      : LightGBM | XGBoost | CatBoost (early stop on the tail of each train window)
    |     +-- Ensemble  : non-negative weights on backtest forecasts
    +-- Final      : refit on all history -> forecast HORIZON -> empirical intervals
         -> pipeline-output/final/ (submission, forecasts, backtests, report)
```

---

# Initialization <a name="2"></a>

---

## Environment Setup

The pipeline runs on CPU. On the demo panel (30 series, about 1,300 days) the full backtest with three boosting models takes a few minutes. Set `USE_GPU = True` on Kaggle to speed up XGBoost and CatBoost on large panels.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn statsmodels pyarrow openpyxl lightgbm xgboost catboost

## Import Libraries

The boosting libraries are imported defensively, so a missing one is skipped and the statistical baselines still produce a complete forecast.

In [ ]:
import os
import re
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.optimize import minimize
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from IPython.display import Markdown, display

AVAILABLE = {}
try:
    import lightgbm as lgb
    AVAILABLE['lightgbm'] = lgb.__version__
except Exception as e:
    AVAILABLE['lightgbm'] = False
    print(f'[warn] lightgbm unavailable: {str(e).strip().splitlines()[0][:100]}')
try:
    import xgboost as xgb
    AVAILABLE['xgboost'] = xgb.__version__
except Exception as e:
    AVAILABLE['xgboost'] = False
    print(f'[warn] xgboost unavailable: {str(e).strip().splitlines()[0][:100]}')
try:
    import catboost
    from catboost import CatBoostRegressor, Pool
    AVAILABLE['catboost'] = catboost.__version__
except Exception as e:
    AVAILABLE['catboost'] = False
    print(f'[warn] catboost unavailable: {str(e).strip().splitlines()[0][:100]}')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('Libraries:', AVAILABLE)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

Paths, columns, the model list, and every forecasting design choice live here. `'auto'` values are resolved in section 4 from the EDA decisions file. `EVENT_DATES` defaults to the Eid al-Fitr (Lebaran) dates for Indonesia, the most influential retail event there. Replace or extend it with the events relevant to the competition, including future dates.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: edit the line for the environment you run in
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>/train.csv'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>/train.csv'
    LOCAL_PATH       = 'demo-data/train.csv'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = 'demo-data/test.csv'
    READ_KWARGS      = {}
    DEMO_IF_MISSING  = True
    EDA_DIR          = 'eda-output'

    # 2) Columns
    DATE_COL   = 'auto'
    TARGET_COL = 'auto'
    ID_COLS    = 'auto'
    EXOG_COLS  = 'auto'
    AGG        = 'sum'

    # 3) Forecast design ('auto' = from EDA decisions)
    FREQ             = 'auto'
    HORIZON          = 'auto'
    SEASONAL_PERIODS = 'auto'
    TRANSFORM        = 'auto'
    FILL_MISSING     = 'auto'
    STRATEGY         = 'direct'
    LAGS             = 'auto'
    ROLL_WINDOWS     = 'auto'
    EVENT_DATES      = ['2021-05-13', '2022-05-02', '2023-04-22', '2024-04-10', '2025-03-31', '2026-03-20', '2027-03-10']
    EVENT_WINDOW     = 14
    N_FOLDS          = 'auto'
    METRIC           = 'auto'
    NON_NEGATIVE     = 'auto'

    # 4) Models
    MODELS           = ['lightgbm', 'xgboost', 'catboost']
    BASELINES        = ['naive', 'seasonal_naive', 'moving_average', 'ets']
    ETS_MAX_SERIES   = 200
    OBJECTIVE        = 'auto'
    WEIGHT_BY_VOLUME = 'auto'
    N_ESTIMATORS     = 3000
    LEARNING_RATE    = 0.03
    EARLY_STOPPING   = 150
    REFIT_IN_FOLD    = True
    MODEL_PARAMS     = {'lightgbm': {}, 'xgboost': {}, 'catboost': {}}
    USE_GPU          = False
    ENSEMBLE         = True
    INTERVAL         = 0.8

    # 5) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'backtests'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Demo Data Generator

The same generator as the EDA notebook, used only when the configured file is missing and `DEMO_IF_MISSING` is true.

In [ ]:
def make_demo(train_path, test_path, seed=42):
    rng = np.random.RandomState(seed)
    dates = pd.date_range('2021-01-01', '2024-06-30', freq='D')
    future = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=28, freq='D')
    lebaran = pd.to_datetime(['2021-05-13', '2022-05-02', '2023-04-22', '2024-04-10'])
    all_dates = dates.append(future)
    t = np.arange(len(all_dates))
    weekly = np.array([0.9, 0.85, 0.9, 0.95, 1.1, 1.35, 1.25])[all_dates.dayofweek]
    yearly = 1 + 0.15 * np.sin(2 * np.pi * (all_dates.dayofyear - 80) / 365.25)
    days_to_eid = np.min(np.abs((all_dates.values[:, None] - lebaran.values[None, :]).astype('timedelta64[D]').astype(int)), axis=1)
    eid = 1 + 1.2 * np.exp(-days_to_eid / 4.0) * (days_to_eid <= 14)
    rows = []
    for s in range(1, 7):
        for it in range(1, 6):
            base = rng.uniform(20, 120) if it <= 3 else rng.uniform(0.3, 1.5)
            trend = 1 + rng.uniform(-0.1, 0.35) * t / len(t)
            price0 = rng.uniform(10, 60)
            price = price0 * (1 + 0.04 * np.sin(t / 90 + s)) * np.where(rng.rand(len(t)) < 0.03, 0.85, 1.0)
            promo = (rng.rand(len(t)) < 0.06).astype(int)
            lam = base * trend * weekly * yearly * eid * (1 + 0.5 * promo) * (price / price0) ** -1.5
            if s == 3 and it == 1:
                lam = lam * np.where(all_dates >= '2023-03-01', 1.6, 1.0)
            y = rng.poisson(lam * rng.gamma(20, 1 / 20, len(t))).astype(float)
            frame = pd.DataFrame({'date': all_dates, 'store': f'S{s}', 'item': f'I{it}', 'sales': y,
                                  'price': price.round(2), 'promo': promo})
            if s == 6:
                frame = frame[frame['date'] >= '2022-07-01']
            rows.append(frame)
    full = pd.concat(rows, ignore_index=True)
    train = full[full['date'] <= dates[-1]].copy()
    spikes = train.sample(12, random_state=seed).index
    train.loc[spikes, 'sales'] *= 8
    train = train.drop(train[(train['date'].between('2022-02-10', '2022-02-16')) & (train['store'] == 'S2')].index)
    train = train.drop(train.sample(frac=0.01, random_state=seed).index)
    test = full[full['date'] > dates[-1]].drop(columns='sales')
    Path(train_path).parent.mkdir(parents=True, exist_ok=True)
    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)
    print(f'Demo data written: {train_path} ({len(train):,} rows), {test_path} ({len(test):,} rows)')

## Load Dataset

The history and the optional future frame are loaded with the same resolver as the other templates. Column roles are fixed in section 4, once the EDA decisions are known.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def resolve_path(path):
    if path is None:
        return None
    p = Path(path)
    if p.exists():
        return p
    roots = [Path('/kaggle/input')] if CFG._ON_KAGGLE else [Path('/content')] if CFG._ON_COLAB else []
    candidates = [f for r in roots for f in r.rglob('*') if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    if not candidates:
        return None
    best = max(candidates, key=lambda f: f.stat().st_size)
    print(f'[warn] {p} not found, falling back to the largest file: {best}')
    return best

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

DATA_PATH = resolve_path(CFG.DATA_PATH)
if DATA_PATH is None and CFG.DEMO_IF_MISSING:
    make_demo(CFG.LOCAL_PATH, CFG.TEST_LOCAL_PATH or 'demo-data/test.csv', CFG.SEED)
    DATA_PATH = Path(CFG.LOCAL_PATH)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found. Edit the path lines in Settings.'
TEST_PATH = resolve_path(CFG.TEST_PATH)
raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
print(f'History: {raw.shape[0]:,} rows x {raw.shape[1]} cols' + (f' | future frame: {raw_test.shape[0]:,} rows' if raw_test is not None else ''))

---

# Toolkit <a name="3"></a>

---

The plotting identity matches the other templates. The metric functions work on long frames of actuals and forecasts, so every model, fold, and horizon step is scored identically. MASE uses a per-series scale computed only from data before each forecast origin.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    fig.savefig(root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png', facecolor='white')
    plt.show()
    plt.close(fig)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def date_axis(ax):
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

def bar_labels(ax, fmt='{:,.0f}', size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], padding=3, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

def to_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)), encoding='utf-8')

def metrics(frame):
    y, p = frame['y_true'].values, frame['y_pred'].values
    e = y - p
    denom = np.abs(y).sum()
    out = {'wape': np.abs(e).sum() / denom if denom > 0 else np.nan,
           'mae': np.abs(e).mean(), 'rmse': np.sqrt((e ** 2).mean()),
           'smape': np.mean(2 * np.abs(e) / np.clip(np.abs(y) + np.abs(p), 1e-9, None)),
           'bias': (p.sum() - y.sum()) / denom if denom > 0 else np.nan}
    if 'scale' in frame and 'series' in frame:
        out['mase'] = (frame.assign(ae=np.abs(e)).groupby('series')['ae'].mean() / frame.groupby('series')['scale'].first()).replace([np.inf], np.nan).mean()
    elif 'scale' in frame:
        out['mase'] = float((np.abs(e) / frame['scale'].values).mean())
    return {k: float(v) for k, v in out.items()}

---

# EDA-Driven Decisions <a name="4"></a>

---

Every design choice is resolved here, in priority order: an explicit value in `Settings`, then `eda_decisions.json`, then a value computed from the data. The decision table records the source of each choice, so the methodology chapter of the report can state exactly which EDA finding drove which modelling decision.

In [ ]:
eda_path = Path(CFG.EDA_DIR) / 'eda_decisions.json'
EDA = json.loads(eda_path.read_text()) if eda_path.exists() else {}
print(f'EDA decisions: {eda_path.resolve()} ({"found" if EDA else "not found, computing from data"})')
if EDA and Path(EDA.get('source', '')).name != DATA_PATH.name:
    print(f'[warn] EDA was run on {EDA.get("source")}, not {DATA_PATH.name}')
DECISIONS = []

def decide(name, user_value, eda_key, compute, why=''):
    if user_value != 'auto':
        value, source = user_value, 'user'
    elif eda_key and EDA.get(eda_key) is not None:
        value, source = EDA[eda_key], 'eda'
    else:
        value, source = compute(), 'computed'
    DECISIONS.append({'setting': name, 'value': str(value)[:80], 'source': source, 'why': why})
    return value

def guess_date():
    for c in raw.columns:
        if 'date' in c.lower() or 'time' in c.lower() or c.lower() in ('ds', 'period', 'month', 'week'):
            return c
    return raw.columns[0]

D = decide('DATE_COL', CFG.DATE_COL, 'date_col', guess_date)
Y = decide('TARGET_COL', CFG.TARGET_COL, 'target_col', lambda: [c for c in raw.columns if pd.api.types.is_numeric_dtype(raw[c])][-1])
IDS = decide('ID_COLS', CFG.ID_COLS, 'id_cols', lambda: [c for c in raw.columns if c not in (D, Y) and not pd.api.types.is_numeric_dtype(raw[c])])
IDS = [c for c in IDS if c in raw.columns]

def prep(frame):
    frame = frame.copy()
    frame[D] = pd.to_datetime(frame[D], errors='coerce')
    frame = frame.dropna(subset=[D])
    frame['series'] = frame[IDS].astype(str).agg(' | '.join, axis=1) if IDS else 'total'
    return frame

raw = prep(raw)
raw_test = prep(raw_test) if raw_test is not None else None
EXOG_ALL = [c for c in raw.columns if c not in IDS + [D, Y, 'series'] and pd.api.types.is_numeric_dtype(raw[c])] if CFG.EXOG_COLS == 'auto' else list(CFG.EXOG_COLS)
EXOG_KNOWN = decide('EXOG_KNOWN', 'auto' if CFG.EXOG_COLS == 'auto' else CFG.EXOG_COLS, 'exog_known_future',
                    lambda: [c for c in EXOG_ALL if raw_test is not None and c in raw_test.columns], 'covariates present in the future frame')
EXOG_KNOWN = [c for c in EXOG_KNOWN if c in raw.columns]
EXOG_PAST = [c for c in EXOG_ALL if c not in EXOG_KNOWN]
print(f'date={D!r} target={Y!r} ids={IDS} known covariates={EXOG_KNOWN} past-only={EXOG_PAST}')

## Frequency, Horizon, Seasonality, Transform

Fallback computations mirror the EDA: the frequency from the most common step, the horizon from the future frame, seasonal periods from the frequency, and the transform from the variance-to-level slope across series.

In [ ]:
OFFSET_DEFAULTS = {'min': ([60, 1440], 60), 'h': ([24, 168], 48), 'D': ([7, 365], 28), 'W': ([52], 13),
                   'MS': ([12], 12), 'ME': ([12], 12), 'QS': ([4], 8), 'YS': ([1], 3)}

def compute_freq():
    step = raw.sort_values(['series', D]).groupby('series')[D].diff().dropna().mode().iloc[0]
    days = step / pd.Timedelta(days=1)
    if days < 1:
        hours = max(1, int(round(days * 24)))
        return 'h' if hours == 1 else f'{hours}h'
    if days < 6:
        return 'D'
    if days < 25:
        return 'W-' + ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN'][raw[D].dt.dayofweek.mode().iloc[0]]
    if days < 80:
        return 'MS' if (raw[D].dt.day == 1).mean() > 0.8 else 'ME'
    return 'QS' if days < 300 else 'YS'

FREQ = decide('FREQ', CFG.FREQ, 'freq', compute_freq, 'most common step between timestamps')
FKEY = 'h' if FREQ.endswith('h') else 'W' if FREQ.startswith('W') else FREQ if FREQ in OFFSET_DEFAULTS else 'D'
periods_default, h_default = OFFSET_DEFAULTS.get(FKEY, ([7], 14))
HORIZON = int(decide('HORIZON', CFG.HORIZON, 'horizon',
                     lambda: int(raw_test.groupby('series')[D].nunique().median()) if raw_test is not None else h_default,
                     'steps in the future frame, else a frequency default'))
SEASONAL_PERIODS = [int(p) for p in decide('SEASONAL_PERIODS', CFG.SEASONAL_PERIODS, 'seasonal_periods', lambda: periods_default,
                                           'frequency defaults confirmed by the periodogram')]
M = SEASONAL_PERIODS[0]

def compute_transform():
    if raw[Y].min() < 0:
        return 'none'
    st = raw.groupby('series')[Y].agg(['mean', 'std'])
    st = st[(st['mean'] > 0) & (st['std'] > 0)]
    if len(st) >= 3:
        slope = np.polyfit(np.log(st['mean']), np.log(st['std']), 1)[0]
        return 'log1p' if slope > 0.75 else 'sqrt' if slope > 0.3 else 'none'
    return 'log1p' if raw[Y].skew() > 2 else 'none'

TRANSFORM = decide('TRANSFORM', CFG.TRANSFORM, 'transform', compute_transform, 'variance-to-level slope across series')
NON_NEG = bool(decide('NON_NEGATIVE', CFG.NON_NEGATIVE, None, lambda: bool(raw[Y].min() >= 0), 'target never negative'))
ZERO_SHARE = float(EDA.get('zero_share', (raw[Y] == 0).mean()))
INTER_SHARE = float(EDA.get('intermittent_share', 0.0))
FILL = decide('FILL_MISSING', CFG.FILL_MISSING, 'fill_missing', lambda: 'zero' if ZERO_SHARE > 0.05 else 'interpolate',
              'zeros are common, so a missing record most likely means no activity')
METRIC = decide('METRIC', CFG.METRIC, 'metric', lambda: 'wape' if ZERO_SHARE > 0.01 else 'smape', 'zero share decides WAPE vs sMAPE')

## Features, Validation, and Loss

Lags come from the EDA autocorrelation analysis: with the direct strategy only lags of at least the horizon are allowed (all information must exist at the forecast origin), while the recursive strategy may use short lags. The objective follows the target treatment: squared error on a log-transformed target, Tweedie for raw zero-heavy targets, squared error otherwise. When the metric is volume-weighted (WAPE) and the target is log-transformed, rows are weighted by series volume so that the loss matches the metric.

In [ ]:
STRATEGY = CFG.STRATEGY
MIN_LAG = HORIZON if STRATEGY == 'direct' else 1

def compute_lags():
    base = [MIN_LAG, MIN_LAG + 1, MIN_LAG + 2] + [k * M for k in range(1, 9) if k * M >= MIN_LAG][:4]
    base += [p for p in SEASONAL_PERIODS[1:] if p >= MIN_LAG]
    return sorted(set(base))

eda_lags = 'lags_direct' if STRATEGY == 'direct' else 'lags_recursive'
LAGS = sorted({int(l) for l in decide('LAGS', CFG.LAGS, eda_lags, compute_lags, f'significant ACF lags usable with the {STRATEGY} strategy') if int(l) >= MIN_LAG})
ROLL_WINDOWS = sorted({int(w) for w in decide('ROLL_WINDOWS', CFG.ROLL_WINDOWS, 'rolling_windows',
                                               lambda: [M, 2 * M, 4 * M] + SEASONAL_PERIODS[1:], 'multiples of the main season')})
EVENTS = pd.to_datetime(CFG.EVENT_DATES) if CFG.EVENT_DATES else pd.DatetimeIndex([])
if EDA.get('special_dates'):
    print('EDA special-day candidates (check they are covered by EVENT_DATES):', EDA['special_dates'][:8])

def compute_folds():
    min_len = int(raw.groupby('series').size().median())
    return int(max(1, min(5, (min_len - 2 * M - HORIZON) // HORIZON)))

N_FOLDS = int(decide('N_FOLDS', CFG.N_FOLDS, None, lambda: int(EDA['cv']['n_folds']) if EDA.get('cv') else compute_folds(),
                     'expanding-window folds that fit in the median series length'))
OBJECTIVE = decide('OBJECTIVE', CFG.OBJECTIVE, None,
                   lambda: 'l2' if TRANSFORM in ('log1p', 'sqrt') else ('tweedie' if NON_NEG and (ZERO_SHARE > 0.1 or INTER_SHARE > 0.2) else 'l2'),
                   'transformed target -> L2; raw zero-heavy target -> Tweedie')
WEIGHT_BY_VOLUME = bool(decide('WEIGHT_BY_VOLUME', CFG.WEIGHT_BY_VOLUME, None, lambda: bool(METRIC == 'wape' and TRANSFORM != 'none'),
                               'align a log-scale loss with the volume-weighted metric'))
MODELS = [m for m in CFG.MODELS if AVAILABLE.get(m)]
SKIPPED = [m for m in CFG.MODELS if m not in MODELS]
DECISIONS.append({'setting': 'MODELS', 'value': str(MODELS), 'source': 'user', 'why': f'skipped (not installed): {SKIPPED}' if SKIPPED else 'all requested models available'})
if EDA.get('model_families'):
    print('EDA model families:', EDA['model_families'])

decision_table = pd.DataFrame(DECISIONS).set_index('setting')
decision_table.to_csv(CFG.FINAL_DIR / 'decisions.csv')
display(decision_table)

#### Insights

> *Fill in after running.* Quote the decision table: which choices came from the EDA, which were overridden, and why. This is the bridge between the EDA chapter and the modelling chapter of the report.

---

# Data Preparation <a name="5"></a>

---

Each series is placed on a regular grid from its first observation to the end of the history, and the future frame is appended for the next `HORIZON` steps. Missing target values inside the lifespan are filled as decided (zero or interpolation), known covariates are forward and backward filled within each series, and the transform is applied. The untransformed target is kept for scoring.

In [ ]:
agg_map = {Y: CFG.AGG, **{c: 'mean' for c in EXOG_ALL}}
hist = raw.groupby(['series', D], as_index=False).agg(agg_map)
ids_map = raw.drop_duplicates('series').set_index('series')[IDS] if IDS else pd.DataFrame(index=hist['series'].unique())
LAST = hist[D].max()
FUTURE_DATES = pd.date_range(LAST, periods=HORIZON + 1, freq=FREQ)[1:]

frames = []
for s, g in hist.groupby('series'):
    grid = pd.date_range(g[D].min(), FUTURE_DATES[-1], freq=FREQ)
    f = g.set_index(D).reindex(grid)
    f.index.name = D
    f['series'] = s
    frames.append(f.reset_index())
data = pd.concat(frames, ignore_index=True)
data['is_future'] = data[D] > LAST
if raw_test is not None and EXOG_KNOWN:
    fut = raw_test.groupby(['series', D], as_index=False)[EXOG_KNOWN].mean()
    data = data.merge(fut, on=['series', D], how='left', suffixes=('', '_future'))
    for c in EXOG_KNOWN:
        data[c] = data[c].fillna(data[f'{c}_future'])
        data = data.drop(columns=f'{c}_future')

gap = data[Y].isna() & ~data['is_future']
if FILL == 'zero':
    data.loc[gap, Y] = 0.0
else:
    data[Y] = data.groupby('series')[Y].transform(lambda s: s.interpolate(limit_direction='both'))
    data.loc[data['is_future'], Y] = np.nan
for c in EXOG_ALL:
    data[c] = data.groupby('series')[c].transform(lambda s: s.ffill().bfill())
for c in IDS:
    data[c] = data['series'].map(ids_map[c])

def fwd(x):
    return np.log1p(np.clip(x, 0, None)) if TRANSFORM == 'log1p' else np.sqrt(np.clip(x, 0, None)) if TRANSFORM == 'sqrt' else x

def inv(x):
    x = np.expm1(x) if TRANSFORM == 'log1p' else np.square(np.clip(x, 0, None)) if TRANSFORM == 'sqrt' else x
    return np.clip(x, 0, None) if NON_NEG else x

data = data.sort_values(['series', D]).reset_index(drop=True)
data['y_raw'] = data[Y]
data['y'] = fwd(data[Y])
SERIES = data['series'].unique().tolist()
print(f'{len(SERIES)} series | {len(data):,} grid rows ({int(gap.sum()):,} gaps filled with {FILL}) | '
      f'history to {LAST.date()} | future {FUTURE_DATES[0].date()} to {FUTURE_DATES[-1].date()}')
display(data.head())

---

# Feature Engineering <a name="6"></a>

---

Features are computed per series on the transformed target. Every target-derived feature is shifted by at least `MIN_LAG` steps (the horizon for the direct strategy), so a row never sees a value that would be unknown at the forecast origin. The feature families are as follows.

| Family | Features | Source of the design |
|---|---|---|
| Lags | `lag_k` for each k in `LAGS` | EDA ACF analysis |
| Rolling | mean and std over `ROLL_WINDOWS`, EWM (alpha 0.1, 0.3) | EDA seasonal periods |
| Same season | mean of the last values at multiples of the main period | EDA seasonal strength |
| Intermittency | zero share in recent windows, steps since last non-zero (only when zeros are common) | EDA demand classes |
| Calendar | hour, weekday, day of month, week, month, year, month start/end flags | EDA seasonal profiles |
| Events | days to next / since last event, in-window flag | `EVENT_DATES` + EDA special days |
| Covariates | known covariates as is plus relative to their recent mean; past-only ones lagged | EDA driver analysis |
| Identity | series id and id columns (categorical), series age | panel structure |

In [ ]:
INTERMITTENT_FEATURES = ZERO_SHARE > 0.05 or INTER_SHARE > 0.1

def event_features(dates):
    out = pd.DataFrame(index=dates.index)
    if len(EVENTS) == 0:
        return out
    ev = EVENTS.sort_values().values
    d = dates.values.astype('datetime64[D]')
    pos = np.searchsorted(ev.astype('datetime64[D]'), d)
    nxt = np.where(pos < len(ev), (ev[np.minimum(pos, len(ev) - 1)].astype('datetime64[D]') - d).astype(int), 9999)
    prv = np.where(pos > 0, (d - ev[np.maximum(pos - 1, 0)].astype('datetime64[D]')).astype(int), 9999)
    out['days_to_event'] = np.clip(nxt, 0, 60)
    out['days_since_event'] = np.clip(prv, 0, 60)
    out['event_window'] = ((nxt <= CFG.EVENT_WINDOW) | (prv <= CFG.EVENT_WINDOW)).astype(int)
    return out

def build_features(frame):
    KEYS = frame['series'].values
    g = frame.groupby('series', sort=False)['y']
    feats = {}
    for L in LAGS:
        feats[f'lag_{L}'] = g.shift(L)
    base = g.shift(MIN_LAG)
    gb = base.groupby(KEYS)
    for w in ROLL_WINDOWS:
        r = gb.rolling(w, min_periods=max(1, w // 2))
        feats[f'roll_mean_{w}'] = r.mean().reset_index(level=0, drop=True)
        if w <= 4 * M:
            feats[f'roll_std_{w}'] = r.std().reset_index(level=0, drop=True)
    for a in (0.1, 0.3):
        feats[f'ewm_{a}'] = gb.transform(lambda s: s.ewm(alpha=a, ignore_na=True).mean())
    ks = [k * M for k in range(1, 9) if k * M >= MIN_LAG][:4]
    if M > 1 and ks:
        feats['same_season_mean'] = pd.concat([g.shift(k) for k in ks], axis=1).mean(axis=1)
    if INTERMITTENT_FEATURES:
        nz = (base > 0).astype(float).where(base.notna())
        for w in sorted({M, 4 * M}):
            feats[f'nonzero_share_{w}'] = nz.groupby(KEYS).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        idx = pd.Series(np.where(base > 0, np.arange(len(frame)), np.nan), index=frame.index)
        feats['steps_since_nonzero'] = np.arange(len(frame)) - idx.groupby(KEYS).ffill().values
    dt = frame[D]
    if FKEY in ('min', 'h'):
        feats['hour'] = dt.dt.hour
    if FKEY in ('min', 'h', 'D'):
        feats['dow'] = dt.dt.dayofweek
        feats['dom'] = dt.dt.day
        feats['is_month_start'] = dt.dt.is_month_start.astype(int)
        feats['is_month_end'] = dt.dt.is_month_end.astype(int)
    if FKEY in ('min', 'h', 'D', 'W'):
        feats['week'] = dt.dt.isocalendar().week.astype(int).values
    feats['month'] = dt.dt.month
    feats['year'] = dt.dt.year
    feats['series_age'] = frame.groupby('series').cumcount()
    for c in EXOG_KNOWN:
        feats[c] = frame[c]
        if frame[c].nunique() > 2:
            past = frame.groupby('series')[c].shift(1)
            ref = past.groupby(KEYS).rolling(4 * M, min_periods=1).mean().reset_index(level=0, drop=True)
            feats[f'{c}_rel'] = frame[c] / ref.replace(0, np.nan)
    for c in EXOG_PAST:
        feats[f'{c}_lag'] = frame.groupby('series')[c].shift(MIN_LAG)
    out = pd.DataFrame(feats, index=frame.index)
    out = out.join(event_features(dt))
    for c in IDS + ['series']:
        out[f'id_{c}'] = pd.Categorical(frame[c].astype(str), categories=sorted(data[c].astype(str).unique())) if c in frame else np.nan
    return out

t0 = time.time()
FEAT = build_features(data)
FEATURES = FEAT.columns.tolist()
CAT_FEATURES = [c for c in FEATURES if c.startswith('id_')]
data['valid'] = FEAT[f'lag_{LAGS[0]}'].notna() if LAGS else True

def family(c):
    for k, v in (('lag_', 'lags'), ('roll_', 'rolling'), ('ewm_', 'rolling'), ('same_season', 'same season'), ('nonzero', 'intermittency'),
                 ('steps_since', 'intermittency'), ('event', 'events'), ('days_', 'events'), ('id_', 'identity'), ('series_age', 'identity')):
        if c.startswith(k):
            return v
    if c in EXOG_KNOWN or c.endswith('_rel') or c.endswith('_lag'):
        return 'covariates'
    return 'calendar'

inv_tbl = pd.Series([family(c) for c in FEATURES], index=FEATURES).value_counts().rename('features').to_frame()
print(f'{len(FEATURES)} features built in {time.time() - t0:.1f}s (strategy {STRATEGY}, min lag {MIN_LAG}, intermittency features: {INTERMITTENT_FEATURES})')
display(inv_tbl)
pd.DataFrame({'feature': FEATURES, 'family': [family(c) for c in FEATURES]}).to_csv(CFG.FINAL_DIR / 'feature_inventory.csv', index=False)

#### Insights

> *Fill in after running.* Which feature families were built, and which EDA finding motivated each one? Mention the event calendar used and the reason behind the minimum lag.

---

# Backtesting Design <a name="7"></a>

---

Forecasts are validated with an expanding-window backtest that mimics deployment: at each cutoff the model sees only the past and forecasts the next `HORIZON` steps. Folds are placed at the end of the history, one horizon apart, because the most recent behaviour is the most relevant. Only originally observed values are scored, never filled gaps. MASE scales are computed per series from data before each cutoff.

In [ ]:
data['observed'] = raw.groupby(['series', D])[Y].sum().reindex(pd.MultiIndex.from_frame(data[['series', D]])).notna().values
hist_dates = np.sort(data.loc[~data['is_future'], D].unique())
CUTOFFS = [pd.Timestamp(hist_dates[-(k * HORIZON) - 1]) for k in range(N_FOLDS, 0, -1)]
STEP = {d: i for i, d in enumerate(pd.date_range(hist_dates[0], FUTURE_DATES[-1], freq=FREQ))}
data['t_idx'] = data[D].map(STEP)
WIDE = data[~data['is_future']].pivot(index=D, columns='series', values='y_raw')

def scales(cutoff):
    h = WIDE.loc[:cutoff]
    s = (h - h.shift(M)).abs().mean()
    return s.where(s > 0, h.diff().abs().mean()).fillna(1.0).clip(lower=1e-6)

def val_mask(cutoff):
    t = STEP[pd.Timestamp(cutoff)]
    return (data['t_idx'] > t) & (data['t_idx'] <= t + HORIZON) & ~data['is_future']

fold_tbl = pd.DataFrame([{'fold': k + 1, 'cutoff': c.date(), 'train_rows': int(((data[D] <= c) & data['valid']).sum()),
                          'valid_rows': int((val_mask(c) & data['observed']).sum())} for k, c in enumerate(CUTOFFS)]).set_index('fold')
display(fold_tbl)

fig, ax = plt.subplots(figsize=(15, 0.5 * N_FOLDS + 1.8))
start = pd.Timestamp(hist_dates[0])
for k, c in enumerate(CUTOFFS):
    end = data.loc[val_mask(c), D].max()
    ax.barh(k, mdates.date2num(c) - mdates.date2num(start), left=mdates.date2num(start), color=SOFT, height=0.6)
    ax.barh(k, mdates.date2num(end) - mdates.date2num(c), left=mdates.date2num(c), color=ACCENT, height=0.6)
ax.barh(N_FOLDS, mdates.date2num(FUTURE_DATES[-1]) - mdates.date2num(LAST), left=mdates.date2num(LAST), color=DARK, height=0.6)
ax.set_yticks(range(N_FOLDS + 1))
ax.set_yticklabels([f'fold {k + 1}' for k in range(N_FOLDS)] + ['final forecast'])
ax.xaxis_date()
date_axis(ax)
handles = [mpl.patches.Patch(color=c, label=l) for c, l in ((SOFT, 'train'), (ACCENT, 'validate'), (DARK, 'forecast'))]
ax.legend(handles=handles, loc='upper left', ncol=3)
suptitle(fig, 'Expanding-window backtest', f'{N_FOLDS} folds of {HORIZON} steps at frequency {FREQ}')
save_fig(fig, 'backtest_design', 'final')

---

# Statistical Baselines <a name="8"></a>

---

Four per-series baselines run on every fold. Naive repeats the last value, seasonal naive repeats the last season, moving average forecasts the mean of the last season, and ETS (Holt-Winters exponential smoothing with damped trend and additive seasonality on the transformed target) is fitted when the number of series is manageable. They need no features and set the bar that the gradient boosting models must clear.

In [ ]:
def baseline_forecast(name, y, H):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) == 0:
        return np.zeros(H)
    if name == 'naive' or len(y) < M:
        return np.repeat(y[-1], H)
    if name == 'seasonal_naive':
        return np.resize(y[-M:], H)
    if name == 'moving_average':
        return np.repeat(y[-M:].mean(), H)
    try:
        yy = fwd(y[-min(len(y), 20 * M + 2 * H):])
        seasonal = 'add' if M > 1 and len(yy) >= 2 * M + 2 else None
        fit = ExponentialSmoothing(yy, trend='add', damped_trend=True, seasonal=seasonal,
                                   seasonal_periods=M if seasonal else None, initialization_method='estimated').fit()
        return inv(fit.forecast(H))
    except Exception:
        return np.resize(y[-M:], H)

BASELINES = [b for b in CFG.BASELINES if b != 'ets' or len(SERIES) <= CFG.ETS_MAX_SERIES]
BT = {}
t0 = time.time()
for name in BASELINES:
    rows = []
    for k, c in enumerate(CUTOFFS):
        sc = scales(c)
        vm = data[val_mask(c)]
        for s, g in vm.groupby('series'):
            hist_s = WIDE[s].loc[:c].dropna()
            if len(hist_s) == 0:
                continue
            fc = baseline_forecast(name, hist_s.values, len(g))
            rows.append(pd.DataFrame({'series': s, D: g[D].values, 'fold': k + 1, 'h': np.arange(1, len(g) + 1),
                                      'y_true': g['y_raw'].values, 'y_pred': np.clip(fc, 0, None) if NON_NEG else fc,
                                      'scale': sc[s], 'observed': g['observed'].values}))
    bt = pd.concat(rows, ignore_index=True)
    BT[name] = bt[bt['observed']].drop(columns='observed').reset_index(drop=True)
    print(f'  [{name}] done ({time.time() - t0:.0f}s)')

def summarize(bt):
    per_fold = pd.DataFrame([metrics(g) for _, g in bt.groupby('fold')])
    return per_fold.mean().to_dict(), per_fold

rows = []
for name, bt in BT.items():
    mean, per_fold = summarize(bt)
    rows.append({'model': name, **mean, f'{METRIC}_std': per_fold[METRIC].std()})
base_lb = pd.DataFrame(rows).set_index('model').sort_values(METRIC)
BEST_BASELINE = base_lb.index[0]
base_lb.to_csv(CFG.BASELINE_DIR / 'baseline_leaderboard.csv')
for name, bt in BT.items():
    bt.to_csv(CFG.BASELINE_DIR / f'backtest_{name}.csv', index=False)
display(base_lb.round(4))
print(f'Best baseline: {BEST_BASELINE} ({METRIC} {base_lb.loc[BEST_BASELINE, METRIC]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
axes[0].barh(base_lb.index[::-1], base_lb[METRIC][::-1], xerr=base_lb[f'{METRIC}_std'][::-1],
             color=[ACCENT if m == BEST_BASELINE else PRIMARY for m in base_lb.index[::-1]], error_kw=dict(ecolor=DARK, lw=1, capsize=3))
bar_labels(axes[0], '{:.3f}')
axes[0].set_title(f'Backtest {METRIC} (mean over folds, whiskers = std)', loc='left')
for i, (name, bt) in enumerate(BT.items()):
    byh = bt.groupby('h').apply(lambda g: metrics(g)[METRIC])
    axes[1].plot(byh.index, byh.values, marker='o', ms=3, lw=1.8, color=PALETTE[i % 10], label=name)
axes[1].set_xlabel('steps ahead')
axes[1].set_ylabel(METRIC)
axes[1].legend(fontsize=8)
axes[1].set_title('Error by forecast step', loc='left')
suptitle(fig, 'Statistical baselines', f'{N_FOLDS} folds x {HORIZON} steps x {len(SERIES)} series')
save_fig(fig, 'baseline_backtest', 'baseline')

#### Insights

> *Fill in after running.* Which baseline wins, and how does its error grow with the forecast step? A strong seasonal naive means seasonality dominates. A strong moving average means the series are noisy around a stable level.

---

# Gradient Boosting Forecasters <a name="9"></a>

---

A single global model per library is trained on all series. Inside each fold, the last `HORIZON` steps of the training window serve as an early-stopping set (never the validation window, which would leak), the best iteration is recorded, and the model is refitted on the full training window with that many trees. The direct strategy predicts the whole validation window in one pass. The recursive strategy predicts one step at a time and feeds each prediction back into the lag features. The objective and sample weights come from section 4.

In [ ]:
class GBM:
    def __init__(self, kind, n_iter=None):
        self.kind, self.n_iter = kind, n_iter

    def _params(self, n):
        tw = OBJECTIVE == 'tweedie'
        if self.kind == 'lightgbm':
            p = dict(n_estimators=n, learning_rate=CFG.LEARNING_RATE, num_leaves=63, min_child_samples=30,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=CFG.SEED,
                     n_jobs=-1, verbose=-1, objective='tweedie' if tw else 'regression')
            if tw:
                p['tweedie_variance_power'] = 1.3
        elif self.kind == 'xgboost':
            p = dict(n_estimators=n, learning_rate=CFG.LEARNING_RATE, max_depth=7, min_child_weight=5, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', enable_categorical=True, random_state=CFG.SEED,
                     n_jobs=-1, objective='reg:tweedie' if tw else 'reg:squarederror')
            if tw:
                p['tweedie_variance_power'] = 1.3
            if CFG.USE_GPU:
                p['device'] = 'cuda'
        else:
            p = dict(iterations=n, learning_rate=CFG.LEARNING_RATE * 2, depth=7, l2_leaf_reg=3.0, random_seed=CFG.SEED, verbose=0,
                     allow_writing_files=False, thread_count=-1, loss_function='Tweedie:variance_power=1.3' if tw else 'RMSE')
            if CFG.USE_GPU:
                p['task_type'] = 'GPU'
        p.update(CFG.MODEL_PARAMS.get(self.kind, {}))
        return p

    def _prep(self, X):
        if self.kind != 'catboost':
            return X
        X = X.copy()
        for c in CAT_FEATURES:
            X[c] = X[c].astype(str)
        return X

    def fit(self, X, y, w=None, X_es=None, y_es=None, w_es=None):
        n = self.n_iter or CFG.N_ESTIMATORS
        Xp = self._prep(X)
        es = X_es is not None and self.n_iter is None
        if self.kind == 'lightgbm':
            self.model = lgb.LGBMRegressor(**self._params(n))
            kw = dict(eval_set=[(X_es, y_es)], eval_sample_weight=[w_es], callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False)]) if es else {}
            self.model.fit(Xp, y, sample_weight=w, **kw)
            self.best = self.model.best_iteration_ if es and self.model.best_iteration_ else n
        elif self.kind == 'xgboost':
            p = self._params(n)
            if es:
                p['early_stopping_rounds'] = CFG.EARLY_STOPPING
            self.model = xgb.XGBRegressor(**p)
            self.model.fit(Xp, y, sample_weight=w, eval_set=[(X_es, y_es)] if es else None,
                           sample_weight_eval_set=[w_es] if es and w_es is not None else None, verbose=False)
            self.best = (self.model.best_iteration + 1) if es else n
        else:
            p = self._params(n)
            if es:
                p['early_stopping_rounds'] = CFG.EARLY_STOPPING
            self.model = CatBoostRegressor(**p)
            self.model.fit(Pool(Xp, y, cat_features=CAT_FEATURES, weight=w),
                           eval_set=Pool(self._prep(X_es), y_es, cat_features=CAT_FEATURES, weight=w_es) if es else None,
                           use_best_model=es)
            self.best = (self.model.get_best_iteration() + 1) if es else n
        return self

    def predict(self, X):
        return np.asarray(self.model.predict(self._prep(X)), dtype=float)

    def importance(self):
        if self.kind == 'lightgbm':
            v = self.model.booster_.feature_importance('gain')
        elif self.kind == 'xgboost':
            v = pd.Series(self.model.get_booster().get_score(importance_type='total_gain')).reindex(FEATURES).fillna(0).values
        else:
            v = self.model.get_feature_importance()
        s = pd.Series(v, index=FEATURES, dtype=float)
        return s / max(s.sum(), 1e-12)

    def contributions(self, X):
        if self.kind == 'lightgbm':
            return np.asarray(self.model.predict(X, pred_contrib=True))[:, :-1]
        if self.kind == 'xgboost':
            return self.model.get_booster().predict(xgb.DMatrix(X, enable_categorical=True), pred_contribs=True)[:, :-1]
        return self.model.get_feature_importance(Pool(self._prep(X), cat_features=CAT_FEATURES), type='ShapValues')[:, :-1]

def volume_weights(mask, cutoff):
    if not WEIGHT_BY_VOLUME:
        return None
    vol = WIDE.loc[:cutoff].tail(4 * max(M, HORIZON)).mean().clip(lower=1e-3)
    w = data.loc[mask, 'series'].map(vol).values
    return w / w.mean()

## Recursive Forecasting Helper

For the recursive strategy, the target after the cutoff is hidden, then each step is predicted and written back before the features of the next step are rebuilt. Only a recent tail of each series is used to rebuild features, which keeps the loop fast.

In [ ]:
TAIL = max(LAGS + ROLL_WINDOWS + [4 * M]) + 2 * HORIZON + 5

def recursive_predict(model, cutoff, dates):
    work = data[data['t_idx'] >= STEP[pd.Timestamp(cutoff)] - TAIL].copy()
    work.loc[work[D] > cutoff, 'y'] = np.nan
    out = {}
    for d in dates:
        sub = work[work[D] <= d]
        Xd = build_features(sub).loc[sub[D] == d, FEATURES]
        p = model.predict(Xd)
        work.loc[Xd.index, 'y'] = p
        out.update(dict(zip(Xd.index, p)))
    return pd.Series(out)

## Backtest Loop

Every active model runs over all folds. The backtest predictions, best iterations, gain importances, and the last fold's model (for interpretation) are stored.

In [ ]:
def es_split(cutoff):
    es_cut = pd.Timestamp(hist_dates[np.searchsorted(hist_dates, np.datetime64(cutoff)) - HORIZON])
    return es_cut

def train_mask(cutoff):
    return (data[D] <= cutoff) & data['valid'] & data['y'].notna() & ~data['is_future']

GBM_INFO = {}
for kind in MODELS:
    rows, iters, imps, t0 = [], [], [], time.time()
    for k, c in enumerate(CUTOFFS):
        feat = FEAT if STRATEGY == 'direct' else build_features(data.assign(y=data['y'].where(data[D] <= c)))
        tr = train_mask(c)
        es_cut = es_split(c)
        fit_m, es_m = tr & (data[D] <= es_cut), tr & (data[D] > es_cut)
        m = GBM(kind).fit(feat.loc[fit_m, FEATURES], data.loc[fit_m, 'y'], volume_weights(fit_m, es_cut),
                          feat.loc[es_m, FEATURES], data.loc[es_m, 'y'], volume_weights(es_m, es_cut))
        best = m.best
        if CFG.REFIT_IN_FOLD:
            m = GBM(kind, n_iter=max(best, 50)).fit(feat.loc[tr, FEATURES], data.loc[tr, 'y'], volume_weights(tr, c))
        vm = val_mask(c)
        if STRATEGY == 'direct':
            pred = pd.Series(m.predict(FEAT.loc[vm, FEATURES]), index=data.index[vm])
        else:
            pred = recursive_predict(m, c, sorted(data.loc[vm, D].unique()))
        sc = scales(c)
        v = data.loc[vm].copy()
        v['y_pred'] = inv(pred.reindex(v.index).values)
        v['h'] = v['t_idx'] - STEP[c]
        v = v[v['observed']]
        rows.append(pd.DataFrame({'series': v['series'].values, D: v[D].values, 'fold': k + 1, 'h': v['h'].values,
                                  'y_true': v['y_raw'].values, 'y_pred': v['y_pred'].values, 'scale': v['series'].map(sc).values}))
        iters.append(best)
        imps.append(m.importance())
        print(f'  [{kind}] fold {k + 1}/{N_FOLDS} cutoff {c.date()} best_iter {best} {METRIC} {metrics(rows[-1])[METRIC]:.4f} ({time.time() - t0:.0f}s)')
    BT[kind] = pd.concat(rows, ignore_index=True)
    GBM_INFO[kind] = {'iters': iters, 'importance': pd.concat(imps, axis=1).mean(axis=1), 'last_model': m, 'last_cutoff': c, 'time': time.time() - t0}
    BT[kind].to_csv(CFG.FINAL_DIR / 'backtests' / f'backtest_{kind}.csv', index=False)
if not MODELS:
    note('No boosting library is available, so the pipeline continues with the statistical baselines.')

#### Insights

> *Fill in after running.* How many trees did early stopping choose, and are they stable across folds? Does each boosting model beat the best baseline in every fold?

---

# Ensemble and Model Selection <a name="10"></a>

---

The boosting models and the best baseline are blended with non-negative weights summing to one, fitted on the backtest forecasts by minimising mean absolute error (the numerator of WAPE). Statistical and machine learning forecasts often fail in different situations, so the blend is usually more robust than either. The selected model is the candidate with the lowest mean backtest error across folds.

In [ ]:
KEY = ['series', D, 'fold']
cands = MODELS + [BEST_BASELINE]
WEIGHTS = None
if CFG.ENSEMBLE and len(cands) >= 2:
    merged = BT[cands[0]][KEY + ['h', 'y_true', 'scale']].copy()
    for c in cands:
        merged = merged.merge(BT[c][KEY + ['y_pred']].rename(columns={'y_pred': c}), on=KEY, how='inner')
    P = merged[cands].values
    yt = merged['y_true'].values
    n = len(cands)
    opt = minimize(lambda w: np.abs(yt - P @ w).mean(), np.full(n, 1 / n), method='SLSQP',
                   bounds=[(0, 1)] * n, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
    WEIGHTS = {c: round(float(w), 4) for c, w in zip(cands, opt.x)}
    BT['ensemble'] = merged[KEY + ['h', 'y_true', 'scale']].assign(y_pred=P @ opt.x)
    BT['ensemble'].to_csv(CFG.FINAL_DIR / 'backtests' / 'backtest_ensemble.csv', index=False)
    print('Ensemble weights:', WEIGHTS)

rows = []
for name, bt in BT.items():
    mean, per_fold = summarize(bt)
    rows.append({'model': name, 'type': 'baseline' if name in BASELINES else 'ensemble' if name == 'ensemble' else 'gbm',
                 **mean, f'{METRIC}_std': per_fold[METRIC].std(), 'folds_beating_best_baseline':
                 int((per_fold[METRIC].values < summarize(BT[BEST_BASELINE])[1][METRIC].values).sum()) if name != BEST_BASELINE else np.nan})
LEADERBOARD = pd.DataFrame(rows).set_index('model').sort_values(METRIC)
BEST = LEADERBOARD.index[0]
LEADERBOARD.to_csv(CFG.FINAL_DIR / 'leaderboard.csv')
display(LEADERBOARD.round(4))
gain = (1 - LEADERBOARD.loc[BEST, METRIC] / LEADERBOARD.loc[BEST_BASELINE, METRIC]) * 100
print(f'Selected: {BEST} ({METRIC} {LEADERBOARD.loc[BEST, METRIC]:.4f}), {gain:+.1f}% vs best baseline {BEST_BASELINE}')

lb = LEADERBOARD.iloc[::-1]
fig, axes = plt.subplots(1, 2 if WEIGHTS else 1, figsize=(17 if WEIGHTS else 10, 0.5 * len(lb) + 2.2), squeeze=False)
colors = [ACCENT if m == BEST else MUTED if lb.loc[m, 'type'] == 'baseline' else PRIMARY for m in lb.index]
axes[0, 0].barh(lb.index, lb[METRIC], xerr=lb[f'{METRIC}_std'], color=colors, error_kw=dict(ecolor=DARK, lw=1, capsize=3))
bar_labels(axes[0, 0], '{:.3f}')
axes[0, 0].set_title(f'Backtest {METRIC} (mean over {N_FOLDS} folds)', loc='left')
if WEIGHTS:
    w = pd.Series(WEIGHTS).sort_values()
    axes[0, 1].barh(w.index, w.values, color=PALETTE[2])
    bar_labels(axes[0, 1], '{:.2f}')
    axes[0, 1].set_title('Ensemble weights', loc='left')
suptitle(fig, 'Forecast model leaderboard', f'selected: {BEST} (orange); baselines in grey')
save_fig(fig, 'leaderboard', 'final')

#### Insights

> *Fill in after running.* Which model is selected, by how much does it beat the best baseline, and in how many folds? How is the ensemble weight split between machine learning and statistical forecasts?

---

# Backtest Diagnostics <a name="11"></a>

---

Aggregate scores hide where forecasts fail. Error is broken down by forecast step (does accuracy decay with distance?), by series (which series are hard, and does the model beat the baseline on them?), and by the EDA demand class (smooth versus intermittent). Actual and forecast paths for the largest series on the last fold make the comparison concrete.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for i, name in enumerate(LEADERBOARD.index):
    byh = BT[name].groupby('h').apply(lambda g: metrics(g)[METRIC])
    is_base = name in BASELINES
    axes[0].plot(byh.index, byh.values, lw=2.4 if name == BEST else 1.4, ls='--' if is_base else '-',
                 color=ACCENT if name == BEST else PALETTE[i % 10], label=name, alpha=0.9)
axes[0].set_xlabel('steps ahead')
axes[0].set_ylabel(METRIC)
axes[0].legend(fontsize=8)
axes[0].set_title('Error by forecast step (dashed = baselines)', loc='left')

per_s = pd.DataFrame({n: BT[n].groupby('series').apply(lambda g: metrics(g)[METRIC]) for n in (BEST, BEST_BASELINE)})
vol = WIDE.sum()
axes[1].scatter(per_s[BEST_BASELINE], per_s[BEST], s=20 + 300 * vol.reindex(per_s.index) / vol.max(), color=PRIMARY, alpha=0.7, edgecolor='white')
lim = [0, np.nanmax(per_s.values) * 1.05]
axes[1].plot(lim, lim, color=MUTED, ls='--')
axes[1].set_xlabel(f'{METRIC} of {BEST_BASELINE}')
axes[1].set_ylabel(f'{METRIC} of {BEST}')
share_better = (per_s[BEST] < per_s[BEST_BASELINE]).mean()
axes[1].set_title(f'Per series: {BEST} better on {share_better:.0%} (below diagonal)', loc='left')
suptitle(fig, 'Backtest diagnostics', 'bubble size = series volume')
save_fig(fig, 'backtest_diagnostics', 'final')
per_s.to_csv(CFG.FINAL_DIR / 'error_by_series.csv')

cls_path = Path(CFG.EDA_DIR) / 'tables' / 'intermittency.csv'
if cls_path.exists():
    cls = pd.read_csv(cls_path).set_index('series')['demand_class']
    by_cls = pd.DataFrame({n: BT[n].assign(c=BT[n]['series'].map(cls)).groupby('c').apply(lambda g: metrics(g)[METRIC]) for n in LEADERBOARD.index})
    by_cls.to_csv(CFG.FINAL_DIR / 'error_by_demand_class.csv')
    display(by_cls.round(3))

In [ ]:
last = BT[BEST][BT[BEST]['fold'] == N_FOLDS]
top = WIDE.sum().sort_values(ascending=False).index[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 7.5), squeeze=False)
c = CUTOFFS[-1]
for ax, s in zip(axes.ravel(), top):
    h = WIDE[s].loc[:c].tail(4 * HORIZON)
    ax.plot(h.index, h.values, color=MUTED, lw=1, label='history')
    g = last[last['series'] == s]
    ax.plot(g[D], g['y_true'], color=DARK, lw=1.4, label='actual')
    ax.plot(g[D], g['y_pred'], color=ACCENT, lw=2, label=BEST)
    b = BT[BEST_BASELINE][(BT[BEST_BASELINE]['fold'] == N_FOLDS) & (BT[BEST_BASELINE]['series'] == s)]
    ax.plot(b[D], b['y_pred'], color=PRIMARY, lw=1.4, ls='--', label=BEST_BASELINE)
    ax.axvline(c, color=MUTED, ls=':', lw=1)
    ax.set_title(short(s, 30), loc='left', fontsize=10)
    date_axis(ax)
axes[0, 0].legend(fontsize=8)
suptitle(fig, f'Last backtest fold: actual vs forecast', f'cutoff {c.date()}, top 6 series by volume')
save_fig(fig, 'backtest_paths', 'final')

#### Insights

> *Fill in after running.* Does error grow with the horizon, and where does the model lose to the baseline? Are intermittent series the main weakness? Do forecasts capture the weekly shape and the event peaks, or are they too flat?

---

# Model Interpretation <a name="12"></a>

---

Gain importance (averaged over folds) shows which features the trees rely on, and exact TreeSHAP contributions on the last fold's validation rows show how much each feature moves the forecasts. Importance aggregated by feature family checks the EDA story: if lags and rolling means dominate, history drives the forecast; if events or covariates matter, the business drivers do.

In [ ]:
GBM_RANKED = [m for m in LEADERBOARD.index if m in MODELS]
if not GBM_RANKED:
    note('No boosting model was trained, so there is nothing to interpret.')
else:
    kind = GBM_RANKED[0]
    info = GBM_INFO[kind]
    vm = val_mask(info['last_cutoff'])
    feat_last = FEAT if STRATEGY == 'direct' else build_features(data.assign(y=data['y'].where(data[D] <= info['last_cutoff'])))
    Xs = feat_last.loc[vm, FEATURES]
    Xs = Xs.sample(min(3000, len(Xs)), random_state=CFG.SEED)
    shap_abs = pd.Series(np.abs(info['last_model'].contributions(Xs)).mean(axis=0), index=FEATURES)
    imp = pd.DataFrame({'gain_share': info['importance'], 'mean_abs_shap': shap_abs}).sort_values('mean_abs_shap', ascending=False)
    imp['family'] = [family(c) for c in imp.index]
    imp.to_csv(CFG.FINAL_DIR / 'feature_importance.csv')
    fam = imp.groupby('family')[['gain_share', 'mean_abs_shap']].sum()
    fam = fam / fam.sum()

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={'width_ratios': [1.5, 1]})
    t = imp.head(20).iloc[::-1]
    fam_colors = {f: PALETTE[i % 10] for i, f in enumerate(sorted(fam.index))}
    axes[0].barh(t.index, t['mean_abs_shap'], color=[fam_colors[f] for f in t['family']])
    axes[0].set_title(f'Top 20 features by mean |SHAP| ({kind})', loc='left')
    handles = [mpl.patches.Patch(color=v, label=k) for k, v in fam_colors.items()]
    axes[0].legend(handles=handles, fontsize=8, loc='lower right')
    f2 = fam.sort_values('mean_abs_shap')
    axes[1].barh(f2.index, f2['mean_abs_shap'], color=[fam_colors[f] for f in f2.index])
    bar_labels(axes[1], '{:.0%}')
    axes[1].set_title('Share of impact by feature family', loc='left')
    suptitle(fig, 'What drives the forecasts?', f'TreeSHAP on {len(Xs):,} rows of the last validation window')
    save_fig(fig, 'feature_importance', 'final')

#### Insights

> *Fill in after running.* Which features and families drive the forecasts? Do event and promo features carry weight, confirming the EDA drivers? Unimportant families can be dropped to simplify the model.

---

# Final Forecast <a name="13"></a>

---

Each component of the selected model is refitted on the full history. Boosting models use the mean best iteration from the backtest, increased by 10% because the final training set is longer. Prediction intervals are empirical: the backtest errors of the selected model, scaled by each series' MASE scale and pooled per forecast step, give the `INTERVAL` quantiles, which are then rescaled to each series. This makes the intervals honest (they reflect real out-of-sample errors) and widen naturally with the horizon.

In [ ]:
fut_mask = data['is_future']
FC = {}
components = list(WEIGHTS) if BEST == 'ensemble' else [BEST]
for name in components:
    if name in MODELS:
        n_iter = int(np.mean(GBM_INFO[name]['iters']) * 1.1) + 1
        tr = train_mask(LAST)
        m = GBM(name, n_iter=n_iter).fit(FEAT.loc[tr, FEATURES], data.loc[tr, 'y'], volume_weights(tr, LAST))
        if STRATEGY == 'direct':
            pred = pd.Series(m.predict(FEAT.loc[fut_mask, FEATURES]), index=data.index[fut_mask])
        else:
            pred = recursive_predict(m, LAST, list(FUTURE_DATES))
        FC[name] = pd.Series(inv(pred.reindex(data.index[fut_mask]).values), index=data.index[fut_mask])
        print(f'  [{name}] refitted with {n_iter} trees')
    else:
        vals = []
        for s, g in data[fut_mask].groupby('series'):
            fc = baseline_forecast(name, WIDE[s].dropna().values, len(g))
            vals.append(pd.Series(np.clip(fc, 0, None) if NON_NEG else fc, index=g.index))
        FC[name] = pd.concat(vals)
        print(f'  [{name}] forecast from full history')
final_pred = sum(WEIGHTS[n] * FC[n] for n in components) if BEST == 'ensemble' else FC[BEST]

res = BT[BEST].assign(z=(BT[BEST]['y_true'] - BT[BEST]['y_pred']) / BT[BEST]['scale'])
lo_q, hi_q = (1 - CFG.INTERVAL) / 2, 1 - (1 - CFG.INTERVAL) / 2
qz = res.groupby('h')['z'].quantile([lo_q, hi_q]).unstack()
qz.columns = ['z_lo', 'z_hi']
sc_full = scales(LAST)
forecast = data.loc[fut_mask, ['series', D] + IDS + EXOG_KNOWN].copy()
forecast['h'] = forecast.groupby('series').cumcount() + 1
forecast['forecast'] = final_pred.reindex(forecast.index).values
z = qz.reindex(forecast['h']).ffill().values
s_scale = forecast['series'].map(sc_full).values
forecast['lower'] = forecast['forecast'] + z[:, 0] * s_scale
forecast['upper'] = forecast['forecast'] + z[:, 1] * s_scale
if NON_NEG:
    forecast[['lower', 'upper']] = forecast[['lower', 'upper']].clip(lower=0)
for n in components:
    forecast[f'fc_{n}'] = FC[n].reindex(forecast.index).values
covered = BT[BEST].assign(lo=BT[BEST]['y_pred'] + BT[BEST]['h'].map(qz['z_lo']) * BT[BEST]['scale'],
                          hi=BT[BEST]['y_pred'] + BT[BEST]['h'].map(qz['z_hi']) * BT[BEST]['scale'])
COVERAGE = float(((covered['y_true'] >= covered['lo']) & (covered['y_true'] <= covered['hi'])).mean())
print(f'Forecast rows: {len(forecast):,} | {CFG.INTERVAL:.0%} interval in-sample coverage on backtest: {COVERAGE:.1%}')
display(forecast.head())

## Forecast Plots

The aggregate forecast is shown with the recent history, followed by the largest individual series, each with its prediction interval.

In [ ]:
agg_hist = WIDE.sum(axis=1).tail(6 * HORIZON)
agg_fc = forecast.groupby(D)[['forecast', 'lower', 'upper']].sum()
fig, ax = plt.subplots(figsize=(16, 4.8))
ax.plot(agg_hist.index, agg_hist.values, color=PRIMARY, lw=1.4, label='history')
ax.plot(agg_fc.index, agg_fc['forecast'], color=ACCENT, lw=2.2, label=f'forecast ({BEST})')
ax.fill_between(agg_fc.index, agg_fc['lower'], agg_fc['upper'], color=ACCENT, alpha=0.18, label=f'{CFG.INTERVAL:.0%} interval (sum of bounds)')
ax.axvline(LAST, color=MUTED, ls=':', lw=1)
ax.legend(loc='upper left')
date_axis(ax)
suptitle(fig, f'Aggregate forecast: next {HORIZON} steps', f'{len(SERIES)} series summed')
save_fig(fig, 'forecast_aggregate', 'final')

fig, axes = plt.subplots(2, 3, figsize=(18, 7.5), squeeze=False)
for ax, s in zip(axes.ravel(), top):
    h = WIDE[s].tail(4 * HORIZON)
    f = forecast[forecast['series'] == s]
    ax.plot(h.index, h.values, color=PRIMARY, lw=1)
    ax.plot(f[D], f['forecast'], color=ACCENT, lw=2)
    ax.fill_between(f[D], f['lower'], f['upper'], color=ACCENT, alpha=0.18)
    ax.axvline(LAST, color=MUTED, ls=':', lw=1)
    ax.set_title(short(s, 30), loc='left', fontsize=10)
    date_axis(ax)
suptitle(fig, 'Series forecasts', f'top 6 series by volume, {CFG.INTERVAL:.0%} empirical intervals')
save_fig(fig, 'forecast_series', 'final')

#### Insights

> *Fill in after running.* Does the forecast continue the recent level and weekly shape, and does it anticipate known events and promotions in the future frame? Is the interval coverage close to its nominal level?

---

# Export <a name="14"></a>

---

The submission follows the future frame when one is provided (same rows and keys), otherwise the generated future grid. The best baseline's forecast is also written to the baseline folder as a fallback submission. The resolved configuration, decision table, leaderboard, backtests, and a markdown report complete the export.

In [ ]:
def to_submission(frame, col):
    out = frame[IDS + [D, col]].rename(columns={col: Y}) if IDS else frame[[D, col]].rename(columns={col: Y})
    if raw_test is not None:
        keys = IDS + [D]
        base = raw_test.drop(columns=['series'], errors='ignore')
        out = base.merge(out.assign(**{k: out[k].astype(base[k].dtype) for k in IDS}), on=keys, how='left')
    return out

to_submission(forecast, 'forecast').to_csv(CFG.FINAL_DIR / 'submission.csv', index=False)
forecast.to_csv(CFG.FINAL_DIR / 'forecast_full.csv', index=False)

base_fc = data.loc[fut_mask, ['series', D] + IDS].copy()
vals = []
for s, g in base_fc.groupby('series'):
    fc = baseline_forecast(BEST_BASELINE, WIDE[s].dropna().values, len(g))
    vals.append(pd.Series(np.clip(fc, 0, None) if NON_NEG else fc, index=g.index))
base_fc['forecast'] = pd.concat(vals)
to_submission(base_fc, 'forecast').to_csv(CFG.BASELINE_DIR / 'submission.csv', index=False)
to_json({'best_baseline': BEST_BASELINE, 'metric': METRIC, 'leaderboard': base_lb.round(5).to_dict(orient='index')},
        CFG.BASELINE_DIR / 'metrics.json')

settings = {k: v for k, v in vars(Settings).items() if not k.startswith('__') and not callable(v)}
resolved = {'freq': FREQ, 'horizon': HORIZON, 'seasonal_periods': SEASONAL_PERIODS, 'transform': TRANSFORM,
            'fill': FILL, 'strategy': STRATEGY, 'lags': LAGS, 'roll_windows': ROLL_WINDOWS, 'objective': OBJECTIVE,
            'weight_by_volume': WEIGHT_BY_VOLUME, 'metric': METRIC, 'n_folds': N_FOLDS, 'cutoffs': [str(c.date()) for c in CUTOFFS],
            'models': MODELS, 'skipped': SKIPPED, 'best': BEST, 'weights': WEIGHTS, 'interval': CFG.INTERVAL,
            'interval_coverage_backtest': COVERAGE, 'features': FEATURES, 'libraries': AVAILABLE}
to_json({'settings': settings, 'resolved': resolved, 'eda_decisions': EDA}, CFG.FINAL_DIR / 'config.json')

cols = [METRIC, f'{METRIC}_std', 'mase', 'rmse', 'bias']
fmt = lambda v: f'{v:.4f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else '-'
lines = [f'# Forecasting Report: {DATA_PATH.name}', '', '## Setup',
         f'- {len(SERIES)} series of `{Y}` at frequency **{FREQ}**, history to {LAST.date()}, horizon **{HORIZON}**',
         f'- Strategy **{STRATEGY}**, lags {LAGS}, rolling windows {ROLL_WINDOWS}, transform {TRANSFORM}, objective {OBJECTIVE}',
         f'- Backtest: {N_FOLDS} expanding-window folds (cutoffs {", ".join(str(c.date()) for c in CUTOFFS)}), metric **{METRIC}**', '',
         '## Decisions', '', '| setting | value | source | why |', '|---|---|---|---|']
lines += [f'| {s} | `{r.value}` | {r.source} | {r.why} |' for s, r in decision_table.iterrows()]
lines += ['', '## Leaderboard', '', '| model | ' + ' | '.join(cols) + ' |', '|---' * (len(cols) + 1) + '|']
for mname, r in LEADERBOARD.iterrows():
    lines.append(f'| {mname}{" (selected)" if mname == BEST else ""} | ' + ' | '.join(fmt(r.get(c, np.nan)) for c in cols) + ' |')
lines += ['', '## Result',
          f'- Selected **{BEST}**: backtest {METRIC} {LEADERBOARD.loc[BEST, METRIC]:.4f}, {gain:+.1f}% vs best baseline {BEST_BASELINE} ({LEADERBOARD.loc[BEST_BASELINE, METRIC]:.4f})',
          f'- {CFG.INTERVAL:.0%} empirical interval coverage on the backtest: {COVERAGE:.1%}']
if WEIGHTS:
    lines.append('- Ensemble weights: ' + ', '.join(f'{k} {v:.2f}' for k, v in WEIGHTS.items()))
if GBM_RANKED:
    lines += ['', f'## Top features ({GBM_RANKED[0]}, mean |SHAP|)'] + [f'{i + 1}. `{f}` ({r.family})' for i, (f, r) in enumerate(imp.head(10).iterrows())]
report_md = '\n'.join(lines)
(CFG.FINAL_DIR / 'report.md').write_text(report_md, encoding='utf-8')
display(Markdown(report_md))

The last cell prints the output tree for both stages.

In [ ]:
for stage in (CFG.BASELINE_DIR, CFG.FINAL_DIR):
    print(f'{stage}/')
    for p in sorted(stage.rglob('*')):
        if p.is_file():
            print(f'   {str(p.relative_to(stage)):<45} {p.stat().st_size / 1e3:>10,.1f} KB')

#### Insights

> *Fill in after running.* Summarise the selected model, its backtest error and gain over the best baseline, the drivers it relies on, the interval coverage, and how each design choice traces back to the EDA.